# MiniMax-H3 Inference (DiffSynth-Studio)

Everyday image-to-video generation: load MiniMax-H3-NF4, optionally load a
trained FL2VA LoRA, take a first-frame image + prompt, and generate a
previewable MP4. Verified against DiffSynth-Studio commit
`b8e3811e5c4abd83a44b99b2bcff7ccabdb26a71` — see
`docs/diffsynth_h3_api_notes.md` in the repo.

This notebook is the everyday generation workflow (base H3, or H3 + a LoRA
you've already trained). For training a new LoRA, use
`H3_LoRA_Training.ipynb` instead — that notebook still produces its own
per-checkpoint validation videos during training.

**Note on "sound prompt":** the current `MiniMaxH3Pipeline.__call__` has a
single `prompt` (plus optional `negative_prompt`) — there is no separate
sound/dialogue prompt parameter. Any spoken dialogue should be written
directly into the main prompt (this matches how the official DiffSynth-Studio
examples do it, e.g. `...she is speaking in english: "..."`). This notebook
does not invent a field that doesn't exist upstream.

## 1. Runtime check

In [ ]:
import subprocess, sys, shutil

print("=== nvidia-smi ===")
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30).stdout)
except Exception as e:
    print("nvidia-smi failed:", e)

import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device:", props.name, "| total VRAM GB:", round(props.total_memory / (1024**3), 2))
else:
    print("WARNING: no GPU. In Colab: Runtime > Change runtime type > GPU, then re-run this cell.")

total, used, free = shutil.disk_usage("/")
print("disk free GB:", round(free/(1024**3), 2))


## 2. Environment setup

Mounts Drive (for cached models and trained LoRA checkpoints -- both are
too large/valuable to re-fetch or lose every session), clones/installs
DiffSynth-Studio, and installs Gradio for the UI. Generated videos are
kept local to this Colab session instead (see the "Browse generated
videos" cell near the end) -- nothing you generate here is written to
Drive automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = "/content/drive/MyDrive/minimax-h3"
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
MODELS_CACHE_DIR = os.path.join(DRIVE_ROOT, "models")

# Generated videos stay on the session's local disk, not Drive -- they're
# scratch output to review/download per-session, not something this
# notebook persists for you. They're gone when the runtime resets or
# disconnects, so download anything you want to keep (see the "Browse
# generated videos" cell below, or the Colab file browser on the left).
GENERATIONS_DIR = "/content/generations"

for path in [CHECKPOINTS_DIR, MODELS_CACHE_DIR, GENERATIONS_DIR]:
    os.makedirs(path, exist_ok=True)
    print("ok:", path)

os.environ.setdefault("MODELSCOPE_CACHE", os.path.join(MODELS_CACHE_DIR, "modelscope_cache"))
os.environ.setdefault("HF_HOME", os.path.join(MODELS_CACHE_DIR, "huggingface_cache"))

# DiffSynth-Studio caches to a cwd-relative ./models/<model_id>/... path by
# default, not governed by MODELSCOPE_CACHE/HF_HOME above. Override to an
# absolute, Drive-backed path so this notebook and H3_LoRA_Training.ipynb
# share one model cache regardless of process/cwd, and it survives runtime
# restarts. Verified against commit b8e3811 (core/loader/config.py).
os.environ.setdefault("DIFFSYNTH_MODEL_BASE_PATH", os.path.join(MODELS_CACHE_DIR, "diffsynth_models"))


In [ ]:
REPO_DIR = "/content/DiffSynth-Studio"

def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        if check:
            raise RuntimeError(f"command failed: {' '.join(cmd)}")
    return result

if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "https://github.com/modelscope/DiffSynth-Studio.git", REPO_DIR])
else:
    run(["git", "pull"], cwd=REPO_DIR)

commit_sha = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).stdout.strip()
print("DiffSynth-Studio commit:", commit_sha)

run(["pip", "install", "-e", ".[all]", "--quiet"], cwd=REPO_DIR)
run(["pip", "install", "gradio", "--quiet"], check=True)


## 3. Model setup

Loads the MiniMax-H3-NF4 base checkpoint. Kept in a module-level variable so
switching LoRAs doesn't require reloading the ~14GB of base weights each
time.

In [ ]:
import torch
from diffsynth.pipelines.minimax_h3_audio_video import MiniMaxH3Pipeline, ModelConfig

_loaded_pipe = None
_loaded_variant = None

def get_pipeline(checkpoint_variant="NF4"):
    global _loaded_pipe, _loaded_variant
    if _loaded_pipe is not None and _loaded_variant == checkpoint_variant:
        return _loaded_pipe

    vram_config = dict(
        offload_dtype=torch.bfloat16, offload_device="cpu",
        onload_dtype=torch.bfloat16, onload_device="cpu",
        preparing_dtype=torch.bfloat16, preparing_device="cuda",
        computation_dtype=torch.bfloat16, computation_device="cuda",
    )

    if checkpoint_variant == "NF4":
        model_configs = [
            ModelConfig(model_id="DiffSynth-Studio/MiniMax-H3-NF4", origin_file_pattern=p, **vram_config)
            for p in ["minimax-h3-fl2va-nf4.safetensors", "minimax-h3-text-encoder-nf4.safetensors",
                      "video_vae_nf4.safetensors", "audio_vae_nf4.safetensors"]
        ]
    elif checkpoint_variant == "BF16":
        model_configs = [
            ModelConfig(model_id="MiniMax/MiniMax-H3", origin_file_pattern=p, **vram_config)
            for p in ["FL2VA/transformer/model*.safetensors", "FL2VA/text_encoder/model*.safetensors",
                      "FL2VA/video_vae/source/model.safetensors", "FL2VA/audio_vae/model.safetensors"]
        ]
    else:
        raise ValueError(f"Unknown checkpoint_variant: {checkpoint_variant}")

    print(f"Loading MiniMax-H3 ({checkpoint_variant})... this can take a while on first run.")
    pipe = MiniMaxH3Pipeline.from_pretrained(
        torch_dtype=torch.bfloat16, device="cuda",
        model_configs=model_configs,
        vram_limit=torch.cuda.mem_get_info("cuda")[1] / (1024 ** 3) - 2,
    )
    _loaded_pipe = pipe
    _loaded_variant = checkpoint_variant
    print("Loaded.")
    return pipe

_current_lora_path = None

def set_lora(pipe, lora_path, strength=1.0):
    """lora_path=None or "" clears any loaded LoRA (base H3 inference)."""
    global _current_lora_path
    if _current_lora_path is not None:
        pipe.clear_lora()
        _current_lora_path = None
    if lora_path:
        pipe.load_lora(pipe.dit, lora_path, alpha=strength)
        _current_lora_path = lora_path

def list_lora_checkpoints():
    """Scan Drive for trained LoRA checkpoints, most recent first."""
    found = []
    if not os.path.isdir(CHECKPOINTS_DIR):
        return found
    for experiment_name in sorted(os.listdir(CHECKPOINTS_DIR)):
        exp_dir = os.path.join(CHECKPOINTS_DIR, experiment_name)
        if not os.path.isdir(exp_dir):
            continue
        for fname in sorted(os.listdir(exp_dir)):
            if fname.endswith(".safetensors"):
                found.append(os.path.join(exp_dir, fname))
    return found


## 4. Image-to-video input, prompt, and settings

The Gradio form below is the everyday interface. It supports:

- base H3 inference (no LoRA), and
- LoRA inference (select a trained checkpoint you produced with
  `H3_LoRA_Training.ipynb`).

`num_frames` is restricted to valid H3 values (`17n + 5`).

In [ ]:
import gradio as gr
from PIL import Image
from diffsynth.utils.data.audio_video import write_video_audio
import time

H3_VALID_FRAME_COUNTS = [39, 56, 73, 90, 107, 124]

def generate(
    first_frame_img, prompt, negative_prompt,
    checkpoint_variant, lora_choice, lora_strength,
    height, width, num_frames, num_inference_steps, seed,
):
    if first_frame_img is None:
        raise gr.Error("Upload a first-frame image before generating.")
    if not prompt or not prompt.strip():
        raise gr.Error("Prompt is required.")

    pipe = get_pipeline(checkpoint_variant)
    lora_path = None if lora_choice in (None, "None", "") else lora_choice
    set_lora(pipe, lora_path, strength=lora_strength)

    first_frame = first_frame_img.convert("RGB")
    video, audio = pipe(
        prompt=prompt, negative_prompt=negative_prompt or " ",
        height=int(height), width=int(width), num_frames=int(num_frames),
        num_inference_steps=int(num_inference_steps), seed=int(seed),
        keyframes=[first_frame], keyframe_indices=[0],
    )

    timestamp = time.strftime("%Y%m%d-%H%M%S")
    lora_tag = "no-lora" if lora_path is None else os.path.splitext(os.path.basename(lora_path))[0]
    out_name = f"{timestamp}_{checkpoint_variant}_{lora_tag}_seed{seed}.mp4"
    out_path = os.path.join(GENERATIONS_DIR, out_name)
    write_video_audio(video=video, audio=audio, output_path=out_path, fps=24, audio_sample_rate=pipe.audio_vae.sample_rate)
    print("saved", out_path)
    return out_path

with gr.Blocks(title="MiniMax-H3 Inference") as demo:
    gr.Markdown(
        "## MiniMax-H3 Inference\n"
        "Base H3 generation, or generation with a trained LoRA. "
        "Include any spoken dialogue directly in the prompt text "
        "(there is no separate sound-prompt field in the current pipeline)."
    )
    with gr.Row():
        with gr.Column():
            checkpoint_variant = gr.Dropdown(["NF4", "BF16"], value="NF4", label="Base checkpoint")
            lora_choice = gr.Dropdown(
                choices=["None"] + list_lora_checkpoints(), value="None",
                label="LoRA checkpoint (optional)",
            )
            refresh_btn = gr.Button("Refresh LoRA list")
            lora_strength = gr.Slider(0.0, 2.0, value=1.0, step=0.05, label="LoRA strength")
            first_frame_img = gr.Image(type="pil", label="First-frame image")
            prompt = gr.Textbox(label="Prompt", lines=4, placeholder="Describe the motion (and any spoken dialogue)...")
            negative_prompt = gr.Textbox(label="Negative prompt (optional)", value=" ")
            with gr.Row():
                height = gr.Number(value=480, precision=0, label="Height")
                width = gr.Number(value=832, precision=0, label="Width")
            num_frames = gr.Dropdown(H3_VALID_FRAME_COUNTS, value=124, label="Frame count (17n+5)")
            with gr.Row():
                num_inference_steps = gr.Number(value=50, precision=0, label="Inference steps")
                seed = gr.Number(value=42, precision=0, label="Seed")
            generate_btn = gr.Button("Generate", variant="primary")
        with gr.Column():
            output_video = gr.Video(label="Generated video")

    refresh_btn.click(lambda: gr.update(choices=["None"] + list_lora_checkpoints()), outputs=lora_choice)
    generate_btn.click(
        generate,
        inputs=[first_frame_img, prompt, negative_prompt, checkpoint_variant, lora_choice, lora_strength,
                height, width, num_frames, num_inference_steps, seed],
        outputs=output_video,
    )


## 5. Launch

`share=False` by default (no public link). Set `share=True` only if you
intentionally want a temporary public URL.

In [ ]:
demo.launch(share=False, debug=False)


## 6. Output

Generated videos are written to `/content/generations/` on this Colab
session's local disk (not Drive), named
`<timestamp>_<checkpoint>_<lora>_seed<seed>.mp4`, and previewable directly
in the Gradio UI above. They do not persist past this session -- use the
next cell to browse/preview everything generated so far, or download files
you want to keep via the Colab file browser (folder icon, left sidebar)
before the runtime disconnects.

## 7. Browse generated videos

Run this any time (no need to relaunch Gradio) to list and preview every
video generated so far this session, most recent first. Videos are embedded
inline (base64), since Colab can't serve arbitrary local file paths directly.

In [ ]:
import glob
from IPython.display import display, Video

files = sorted(glob.glob(os.path.join(GENERATIONS_DIR, "*.mp4")), key=os.path.getmtime, reverse=True)
if not files:
    print("No generated videos yet -- run a generation in the Gradio UI above first.")
for f in files:
    size_mb = os.path.getsize(f) / (1024 ** 2)
    print(f"{os.path.basename(f)}  ({size_mb:.1f} MB)")
    display(Video(f, embed=True, width=480))
